In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.models.vgg import vgg16_bn
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
# Importer ZennitCRP
from zennit.composites import EpsilonPlusFlat
from zennit.canonizers import SequentialMergeBatchNorm
from crp.attribution import CondAttribution
from crp.helper import get_layer_names
from crp.concepts import ChannelConcept
from tqdm import tqdm
import time
from torch.utils.data import DataLoader
from pathlib import Path
import json

In [ ]:
# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = 'feature_importance_results'
Path(SAVE_DIR).mkdir(exist_ok=True)

In [ ]:
# Initialisation du modèle VGG16
model = vgg16_bn(pretrained=True).to(device).eval()

In [ ]:
# Transformations d'images
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
cifar10 = torchvision.datasets.CIFAR10(root='./cifar10', train=True, download=True, transform=transform)
cifar10_test = torchvision.datasets.CIFAR10(root='./cifar10', train=False, download=True, transform=transform)

train_loader = DataLoader(cifar10, batch_size=32, shuffle=True)
test_loader = DataLoader(cifar10_test, batch_size=32, shuffle=False)

# Échantillon de référence
sample_img, sample_label = cifar10[0]
sample_img = sample_img.unsqueeze(0).to(device)

In [ ]:
def compute_feature_importance(model, input_tensor, layer_idx, pred_class):
    """Calcule l'importance de chaque feature pour la prédiction."""
    with torch.no_grad():
        # Score original
        original_output = model(input_tensor)
        original_score = torch.nn.functional.softmax(original_output, dim=1)[0, pred_class]
        
        # Stocker les activations
        activations = []
        def hook_fn(module, input, output):
            activations.append(output.clone())
        hook = model.features[layer_idx].register_forward_hook(hook_fn)
        
        # Forward pass pour obtenir les activations
        _ = model(input_tensor)
        hook.remove()
        
        # Calculer les scores pour chaque feature masquée
        importances = []
        for feature_idx in tqdm(range(512), desc="Calculating feature importance"):
            modified_activations = activations[0].clone()
            modified_activations[:, feature_idx] = 0
            
            # Continuer la forward pass depuis la couche modifiée
            x = modified_activations
            for layer in model.features[layer_idx+1:]:
                x = layer(x)
            x = model.avgpool(x)
            x = torch.flatten(x, 1)
            x = model.classifier(x)
            
            new_score = torch.nn.functional.softmax(x, dim=1)[0, pred_class]
            importances.append((original_score - new_score).item())
            
    return np.array(importances)

In [ ]:
def evaluate_model(model, dataloader):
    """Évalue la précision du modèle sur un jeu de données."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [ ]:
def plot_feature_importance_evolution(importances_history, save_path=None):
    """Visualise l'évolution de l'importance des features."""
    plt.figure(figsize=(12, 6))
    
    # Plot des features les plus importantes
    mean_importances = np.mean(importances_history, axis=0)
    top_features = np.argsort(mean_importances)[-5:]
    
    for feature_idx in top_features:
        feature_history = [epoch_imp[feature_idx] for epoch_imp in importances_history]
        plt.plot(range(1, len(importances_history) + 1), 
                feature_history, 
                label=f'Feature {feature_idx}',
                linewidth=2)
    
    plt.xlabel("Époque")
    plt.ylabel("Importance")
    plt.title("Évolution des 5 features les plus importantes")
    plt.legend()
    plt.grid(True)
    
    if save_path:
        plt.savefig(save_path)
    plt.show()
    plt.close()

In [ ]:
def save_results(importances_history, accuracies, save_dir):
    """Sauvegarde les résultats de l'analyse."""
    np.save(f'{save_dir}/importances_history.npy', importances_history)
    with open(f'{save_dir}/accuracies.json', 'w') as f:
        json.dump(accuracies, f)

In [ ]:
# Entraînement et suivi
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

n_epochs = 10
importances_history = []
train_accuracies = []
test_accuracies = []

In [ ]:
for epoch in range(n_epochs):
    print(f"\nEpoch {epoch+1}/{n_epochs}")
    
    # Phase d'entraînement
    model.train()
    for inputs, labels in tqdm(train_loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    # Évaluation
    model.eval()
    train_acc = evaluate_model(model, train_loader)
    test_acc = evaluate_model(model, test_loader)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Test Accuracy: {test_acc:.2f}%")
    
    # Calcul des importances
    importances = compute_feature_importance(model, sample_img, layer_idx=40, pred_class=sample_label)
    importances_history.append(importances)
    
    # Sauvegarde intermédiaire
    save_results(importances_history, 
                {'train': train_accuracies, 'test': test_accuracies},
                SAVE_DIR)
    
    # Visualisation
    plot_feature_importance_evolution(importances_history, 
                                    save_path=f'{SAVE_DIR}/importance_evolution_epoch_{epoch+1}.png')



In [ ]:
# Analyse finale
mean_importances = np.mean(importances_history, axis=0)
top_features = np.argsort(mean_importances)[-10:]  # Top 10 features
print("\nTop 10 features les plus importantes:")
for idx in reversed(top_features):
    print(f"Feature {idx}: {mean_importances[idx]:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Train')
plt.plot(test_accuracies, label='Test')
plt.xlabel('Époque')
plt.ylabel('Précision (%)')
plt.title('Évolution de la précision')
plt.legend()
plt.grid(True)
plt.savefig(f'{SAVE_DIR}/accuracy_evolution.png')
plt.show()

In [ ]:
def analyze_epoch_features(importances_history, epoch_idx):
    """Analyse les features les plus importantes pour une époque donnée."""
    epoch_importances = importances_history[epoch_idx]
    top_indices = np.argsort(epoch_importances)[-10:]  # Top 10 features
    
    print(f"\nTop 10 features les plus importantes à l'époque {epoch_idx + 1}:")
    for idx in reversed(top_indices):
        print(f"Feature {idx}: {epoch_importances[idx]:.4f}")
    
    return top_indices, epoch_importances

# Modification de la visualisation pour inclure le code couleur
plt.figure(figsize=(20, 10))

# Déterminer la nature des features (stable positive, stable négative, ou variable)
initial_values = importances_history[0]
final_values = importances_history[-1]

stable_positive = []
stable_negative = []
variable = []

for feature_idx in range(512):
    if initial_values[feature_idx] > 0 and final_values[feature_idx] > 0:
        stable_positive.append(feature_idx)
    elif initial_values[feature_idx] < 0 and final_values[feature_idx] < 0:
        stable_negative.append(feature_idx)
    else:
        variable.append(feature_idx)

# Tracer les courbes avec le code couleur
for feature_idx in range(512):
    feature_history = [epoch_imp[feature_idx] for epoch_imp in importances_history]
    
    if feature_idx in stable_positive:
        color = 'blue'
    elif feature_idx in stable_negative:
        color = 'red'
    else:
        color = 'orange'
    plt.plot(range(1, n_epochs + 1), feature_history, 
            linewidth=2, label=f'Feature {feature_idx}', color=color)


plt.xlabel("Époque")
plt.ylabel("Importance")
plt.title("Évolution de l'importance des 512 features\n" + 
          "Bleu: stable positive, Rouge: stable négative, Orange: variable\n" +
          "Features importantes en couleur vive")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/all_features_evolution_colored.png', dpi=300, bbox_inches='tight')
plt.show()

# Exemple d'utilisation pour analyser une époque spécifique
epoch_to_analyze = 9# Changer pour l'époque désirée
top_indices, epoch_importances = analyze_epoch_features(importances_history, epoch_to_analyze)

# Afficher des statistiques supplémentaires
print(f"\nStatistiques sur les features:")
print(f"Features stables positives: {len(stable_positive)}")
print(f"Features stables négatives: {len(stable_negative)}")
print(f"Features variables: {len(variable)}")

# Créer un histogramme de la distribution des importances
plt.figure(figsize=(12, 6))
plt.hist(epoch_importances, bins=50, color='skyblue', edgecolor='black')
plt.title(f'Distribution des importances des features à l\'époque {epoch_to_analyze + 1}')
plt.xlabel('Importance')
plt.ylabel('Nombre de features')
plt.savefig(f'{SAVE_DIR}/importance_distribution_epoch_{epoch_to_analyze + 1}.png')
plt.show()

In [ ]:
def compute_feature_importance(model, input_tensor, layer_idx, allFeaturesIDX, pred_class):
    """
    Calcule l'importance globale des features d'une couche donnée pour une classe prédite
    après désactivation de toutes les features spécifiées dans allFeaturesIDX.

    Arguments :
    - model : le modèle VGG16
    - input_tensor : l'image d'entrée sous forme de tenseur
    - layer_idx : l'index de la couche (ex : 40)
    - allFeaturesIDX : liste des indices des features à désactiver (list)
    - pred_class : la classe prédite initialement

    Retourne :
    - La différence entre le score original et le score après désactivation
    """

    # Obtenir la probabilité originale de la classe prédite
    with torch.no_grad():
        output_original = model(input_tensor)
        probs_original = torch.nn.functional.softmax(output_original, dim=1)
        original_score = probs_original[0, pred_class].item()

    def zero_out_features(module, input, output):
        for idx in allFeaturesIDX:
            output[:, idx, :, :] = 0  # Désactiver chaque feature spécifiée
        return output

    # Ajouter un hook temporaire
    hook = model.features[layer_idx].register_forward_hook(zero_out_features)

    # Faire une prédiction avec les features désactivées
    with torch.no_grad():
        output_disabled = model(input_tensor)
        probs_disabled = torch.nn.functional.softmax(output_disabled, dim=1)
        new_score = probs_disabled[0, pred_class].item()

    # Supprimer le hook
    hook.remove()

    # Calcul de l'importance globale des features désactivées
    importance = original_score - new_score
    print(original_score)
    print(new_score)
    print(f"Importance globale après désactivation des features : {importance:.4f}")

    return importance